In [ ]:
# [1] necessery libraries
import random
import numpy as np
import pandas as pd
import simpy
import math

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [ ]:
# [2] defining parameters
BASE_PARAMS = {
    # scenario parameters
    "SIM_TIME": 640,  # 24 hours
    "ARRIVAL_CUTOFF": 640,
    "DRAIN_TIME": 0,
    "WARMUP_MEAN": 120,
    # interarrival time for patients
    "INTERARRIVAL_MEAN": 4,
    # resources
    "N_CLINICIANS": 4,
    "N_BEDS": 15,
    # Decision latency
    "DECISION_LAT_MEAN": 12.0,
    "DECISION_LAT_SD": 2,
    # Treatment time
    "TREAT_BASE_MEAN": 30.0,
    "TREAT_BASE_SD": 5,
    "TREAT_SEVERITY_MULT": 0.6,  # severity factor for treatment time
    "BED_SEVERITY_THRESHOLD": 0.7,  # severity threshold for bed assignment
    # detorioration parameters
    "WAIT_TOL_MEAN": 25.0,
    "WAIT_TOL_SD": 3,
    "DETERIORATION_RATE": 0.03,
}

AI_PARAMS = {
    "AI_PRIORITY_BOOST": 0.15,  # between 0.1 and 0.2
    "AI_EFFECTIVENESS": 0.50,  # between 0.1 and 0.4
}

# just for checking the parameters
print("Input rate: ", 60 / BASE_PARAMS["INTERARRIVAL_MEAN"], "people per hours")
print(
    "Treatment capacity: ",
    BASE_PARAMS["N_BEDS"] * 60 / BASE_PARAMS["TREAT_BASE_MEAN"],
    "people per hours",
)
print(
    "Decision capacity non-AI: ",
    BASE_PARAMS["N_CLINICIANS"] * 60 / BASE_PARAMS["DECISION_LAT_MEAN"],
    "people per hours",
)
print(
    "Decision capacity AI 20%: ",
    (BASE_PARAMS["N_CLINICIANS"] - 1) * 60 / (BASE_PARAMS["DECISION_LAT_MEAN"] * 0.8),
    "people per hours",
)
print(
    "Decision capacity AI 40%: ",
    (BASE_PARAMS["N_CLINICIANS"] - 1) * 60 / (BASE_PARAMS["DECISION_LAT_MEAN"] * 0.6),
    "people per hours",
)
print(
    "Decision capacity AI 60%: ",
    (BASE_PARAMS["N_CLINICIANS"] - 1) * 60 / (BASE_PARAMS["DECISION_LAT_MEAN"] * 0.4),
    "people per hours",
)


In [ ]:
# [3] useful functions used to generate and clamp random variables
# the value is clamped between lo and hi for system stability
def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def pos_normal(mean, sd, min_val=0.2):
    # possitive normal distribution
    x = random.gauss(mean, sd)
    while x <= 0:
        x = random.gauss(mean, sd)
    return max(x, min_val)


# Because patient arrivals are well approximated by a Poisson process in service systems
def exp_interarrival(mean):
    # Exponential distribution for inter-arrival times
    return random.expovariate(1.0 / mean)


In [ ]:
# [4] generating Patient class id, severity, wait tolerance, deterioration rate
"""Patient agents encode heterogeneity in severity and waiting tolerance,
 enabling the model to capture how delays translate into 
deterioration and propagate inefficiencies through the system"""


class Patient:
    def __init__(self, pid, arrival_time):
        self.pid = pid
        self.arrival_time = arrival_time

        # severity must stay between 0.0 and 1.0
        self.severity = clamp(random.random(), 0.0, 1.0)

        # wait tolerance, minimum value is 1.0 minute
        self.wait_tolerance = pos_normal(
            BASE_PARAMS["WAIT_TOL_MEAN"], BASE_PARAMS["WAIT_TOL_SD"], min_val=1.0
        )

        # deterioration rate
        self.deterioration_rate = BASE_PARAMS["DETERIORATION_RATE"]


In [ ]:
# [5]defining functions for decision latency and treatment time
def decision_latency(ai_enabled: bool, queue_len: int, params: dict, ai_params: dict):

    mean = params["DECISION_LAT_MEAN"]
    sd = params["DECISION_LAT_SD"]

    # if it is congested, then increase latency
    congestion_mult = 1.0 + 0.10 * (queue_len // 5)

    if ai_enabled:
        eff = ai_params["AI_EFFECTIVENESS"]  # 0.1 - 0.4
        mean *= 1.0 - eff
        sd *= 1.0 - eff

    return pos_normal(mean * congestion_mult, sd, min_val=0.5)


def treatment_time(patient: Patient, params: dict):

    base = pos_normal(
        params["TREAT_BASE_MEAN"], params["TREAT_BASE_SD"], min_val=1.0
    )  # minimum treatment time is 1.0 minute
    mult = 1.0 + params["TREAT_SEVERITY_MULT"] * patient.severity
    return base * mult


In [ ]:
# [6] for deterioration application
def apply_deterioration(patient: Patient, waited: float):

    excess = max(0.0, waited - patient.wait_tolerance)
    if excess > 0:
        patient.severity = clamp(
            patient.severity + excess * patient.deterioration_rate, 0.0, 1.0
        )


In [ ]:
# [7] the functiuon simulateing patient process through the system
def patient_process(
    env,
    patient: Patient,
    clinicians,
    beds,
    ai_enabled: bool,
    params: dict,
    ai_params: dict,
    logs: list,
):

    # entrance time
    arrival = env.now

    # patient enters clinician queue
    with clinicians.request() as clin_req:
        yield clin_req
        visit_start = env.now

        queue_len = len(clinicians.queue)

        # patient visit / decision time
        visit_time = decision_latency(
            ai_enabled=ai_enabled,
            queue_len=queue_len,
            params=params,
            ai_params=ai_params,
        )
        yield env.timeout(visit_time)

    # end of visit
    visit_end = env.now
    visit_time = visit_end - visit_start
    # total time to end of visit
    total_time = visit_end - arrival
    severity_before = patient.severity
    # refresh severity after waiting for clinician
    waited_for_clinician = visit_start - arrival
    if waited_for_clinician > patient.wait_tolerance:
        apply_deterioration(patient, waited_for_clinician)

    severity_after_visit = patient.severity

    # ------------------------------------------------
    # patient needs bed or not?
    # ------------------------------------------------
    needs_bed = severity_after_visit >= params["BED_SEVERITY_THRESHOLD"]

    wait_for_bed = 0.0
    treat_time = 0.0

    if needs_bed:
        bed_request_time = env.now
        with beds.request() as bed_req:
            yield bed_req
            bed_start = env.now

            wait_for_bed = bed_start - bed_request_time

            # treatment time (only if admitted)
            treat_time = treatment_time(patient, params)
            yield env.timeout(treat_time)

    discharge = env.now
    los = discharge - arrival

    # ------------------------------------------------
    # recording logs
    # ------------------------------------------------
    if env.now >= params.get("warmup", BASE_PARAMS["WARMUP_MEAN"]):
        logs.append(
            {
                "pid": patient.pid,
                "arrival": arrival,
                "visit_start": visit_start,
                "visit_end": visit_end,
                "wait_for_clinician": waited_for_clinician,
                "visit_time": visit_time,
                "severity_after_visit": severity_after_visit,
                "needs_bed": int(needs_bed),
                "wait_for_bed": wait_for_bed,
                "treat_time": treat_time,
                "los": los,
                "ai": ai_enabled,
                "discharge": discharge,
                "decision_latency": visit_time,
                "severity_after": severity_after_visit,
                "queue_len_at_request": queue_len,
                "deteriorated": int(severity_after_visit > severity_before + 1e-9),
            }
        )

In [ ]:
# [8] here we simulate patient arrivals


def arrival_generator(env, clinicians, beds, ai_enabled, params, ai_params, logs):
    pid = 0
    while env.now < params["ARRIVAL_CUTOFF"]:
        pid += 1
        p = Patient(pid, env.now)
        env.process(
            patient_process(
                env, p, clinicians, beds, ai_enabled, params, ai_params, logs
            )
        )
        ia = exp_interarrival(params["INTERARRIVAL_MEAN"])
        yield env.timeout(ia)


In [ ]:
# [9] running the scenario and collecting results (this is the main function)
def run_scenario(ai_enabled: bool, params: dict, ai_params: dict, seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)

    env = simpy.Environment()
    clinicians = simpy.Resource(env, capacity=params["N_CLINICIANS"])
    beds = simpy.Resource(env, capacity=params["N_BEDS"])

    logs = []
    env.process(
        arrival_generator(env, clinicians, beds, ai_enabled, params, ai_params, logs)
    )
    env.run(until=params["ARRIVAL_CUTOFF"] + params["DRAIN_TIME"])

    df = pd.DataFrame(logs)

    # Throughput = number of completed patients / total simulation time
    throughput = len(df) / params["SIM_TIME"] if len(df) else 0.0

    # CV(LOS)
    cv_los = (
        (df["los"].std() / df["los"].mean())
        if len(df) and df["los"].mean() > 0
        else np.nan
    )

    summary = {
        "ai": ai_enabled,
        "n_completed": len(df),
        "throughput_per_min": throughput,
        "los_mean": df["los"].mean() if len(df) else np.nan,
        "los_p95": df["los"].quantile(0.95) if len(df) else np.nan,
        "decision_latency_mean": df["decision_latency"].mean() if len(df) else np.nan,
        "cv_los": cv_los,
        "deterioration_rate": df["deteriorated"].mean() if len(df) else np.nan,
        "wait_for_clinician_mean": df["wait_for_clinician"].mean()
        if len(df)
        else np.nan,
    }
    return df, summary


In [ ]:
# [10] here we run monte carlo simulations for different AI effectiveness levels to compare performance

N_REP = 1000
eff_levels = np.arange(0.10, 0.4, 0.01)
rng = np.random.default_rng(12345)

base_rows = []

for r in range(N_REP):
    seed = int(rng.integers(1, 1e9))

    p_ref = BASE_PARAMS.copy()
    p_ref["N_CLINICIANS"] = BASE_PARAMS["N_CLINICIANS"]

    df_ref, ref_sum = run_scenario(
        ai_enabled=False, params=p_ref, ai_params=AI_PARAMS, seed=seed
    )

    base_rows.append(
        {
            "los_mean": ref_sum["los_mean"],
            "los_p95": ref_sum["los_p95"],
            "throughput": ref_sum["throughput_per_min"],
            "decision_latency": ref_sum["decision_latency_mean"],
        }
    )

df_base_mc = pd.DataFrame(base_rows)

base_metrics = {
    "los_mean_ref": df_base_mc["los_mean"].mean(),
    "los_mean_ref_std": df_base_mc["los_mean"].std(),
    "los_p95_ref": df_base_mc["los_p95"].mean(),
    "throughput_ref": df_base_mc["throughput"].mean(),
    "throughput_ref_std": df_base_mc["throughput"].std(),
    "decision_latency_ref": df_base_mc["decision_latency"].mean(),
}

rows = []

for eff in eff_levels:
    mc_rows = []

    for r in range(N_REP):
        seed = int(rng.integers(1, 1e9))

        p = BASE_PARAMS.copy()
        p["N_CLINICIANS"] = BASE_PARAMS["N_CLINICIANS"] - 1

        a = AI_PARAMS.copy()
        a["AI_EFFECTIVENESS"] = float(eff)

        df, summ = run_scenario(ai_enabled=True, params=p, ai_params=a, seed=seed)

        mc_rows.append(
            {
                "los_mean": summ["los_mean"],
                "los_p95": summ["los_p95"],
                "throughput": summ["throughput_per_min"],
                "decision_latency": summ["decision_latency_mean"],
            }
        )

    mc_df = pd.DataFrame(mc_rows)

    rows.append(
        {
            "eff": eff,
            # Monte Carlo means
            "los_mean": mc_df["los_mean"].mean(),
            "los_p95": mc_df["los_p95"].mean(),
            "throughput": mc_df["throughput"].mean(),
            "decision_latency_mean": mc_df["decision_latency"].mean(),
            # Monte Carlo std (خیلی مهم)
            "los_mean_std": mc_df["los_mean"].std(),
            "throughput_std": mc_df["throughput"].std(),
            # gaps vs baseline
            "gap_los_mean": mc_df["los_mean"].mean() - base_metrics["los_mean_ref"],
            "gap_throughput": mc_df["throughput"].mean()
            - base_metrics["throughput_ref"],
        }
    )

# printing system capacities for checking
print("Input rate: ", 60 / BASE_PARAMS["INTERARRIVAL_MEAN"], "people per hours")
print(
    "Total input: ",
    (BASE_PARAMS["SIM_TIME"] - BASE_PARAMS["WARMUP_MEAN"])
    / BASE_PARAMS["INTERARRIVAL_MEAN"],
    "people",
)
print(
    "Treatment capacity: ",
    BASE_PARAMS["N_BEDS"] * 60 / BASE_PARAMS["TREAT_BASE_MEAN"],
    "people per hours",
)
print(
    "Total Treatment capacity: ",
    BASE_PARAMS["N_BEDS"]
    * (BASE_PARAMS["SIM_TIME"] - BASE_PARAMS["WARMUP_MEAN"])
    / BASE_PARAMS["TREAT_BASE_MEAN"],
    "people",
)
print(
    "Decision capacity non-AI: ",
    BASE_PARAMS["N_CLINICIANS"] * 60 / BASE_PARAMS["DECISION_LAT_MEAN"],
    "people per hours",
)
print(
    "Total Decision capacity non-AI: ",
    BASE_PARAMS["N_CLINICIANS"]
    * (BASE_PARAMS["SIM_TIME"] - BASE_PARAMS["WARMUP_MEAN"])
    / BASE_PARAMS["DECISION_LAT_MEAN"],
    "people",
)
print(
    "Decision capacity AI 20%: ",
    (BASE_PARAMS["N_CLINICIANS"] - 1) * 60 / (BASE_PARAMS["DECISION_LAT_MEAN"] * 0.8),
    "people per hours",
)
print(
    "Decision capacity AI 40%: ",
    (BASE_PARAMS["N_CLINICIANS"] - 1) * 60 / (BASE_PARAMS["DECISION_LAT_MEAN"] * 0.6),
    "people per hours",
)
print(
    "Decision capacity AI 60%: ",
    (BASE_PARAMS["N_CLINICIANS"] - 1) * 60 / (BASE_PARAMS["DECISION_LAT_MEAN"] * 0.4),
    "people per hours",
)

df_sweep_mc = pd.DataFrame(rows)


In [ ]:
print("total visited", df_base_mc["throughput"].mean() * 480)

baseline_row = {
    "eff": 0.0,  # 0% AI
    "los_mean": base_metrics["los_mean_ref"],
    "los_mean_std": base_metrics["los_mean_ref_std"],
    "throughput": base_metrics["throughput_ref"],
    "throughput_std": base_metrics["throughput_ref_std"],
}


summary_table = pd.DataFrame(rows)

summary_table = pd.concat(
    [pd.DataFrame([baseline_row]), summary_table], ignore_index=True
)

# choosing relevant columns
summary_table = summary_table[
    ["eff", "los_mean", "los_mean_std", "throughput", "throughput_std"]
]


summary_table["eff"] = (summary_table["eff"] * 100).astype(int)


summary_table = summary_table.sort_values("eff").reset_index(drop=True)

summary_table = summary_table.round(
    {"los_mean": 2, "los_mean_std": 2, "throughput": 4, "throughput_std": 4}
)

summary_table.columns = [
    "AI Effectiveness (%)",
    "Mean LOS (min)",
    "LOS Std (min)",
    "Mean Throughput (patients/min)",
    "Throughput Std",
]

summary_table


In [ ]:
# [11] finding break-even point
# Break-even: LOS_mean better/equal + LOS_p95 better/equal + throughput better/equal

cond = (
    (df_sweep_mc["los_mean"] <= base_metrics["los_mean_ref"])
    & (df_sweep_mc["los_p95"] <= base_metrics["los_p95_ref"])
    & (df_sweep_mc["throughput"] >= base_metrics["throughput_ref"])
)


break_even = df_sweep_mc[cond].head(1)
if len(break_even) == 0:
    print("No break-even point found in this range.")
else:
    print("Break-even found at eff =", break_even.iloc[0]["eff"])
    break_even


In [ ]:
# [12] defining plotting functions
import matplotlib.pyplot as plt
import numpy as np


def plot_kpi_vs_eff(df_sweep, ref_value, kpi_col, kpi_label, title):
    x = df_sweep["eff"] * 100
    y = df_sweep[kpi_col].values

    plt.figure(figsize=(6, 4))
    plt.plot(x, y, marker="o", label="3 clinicians + AI")

    # four physicians without AI with dashed line
    plt.axhline(ref_value, linestyle="--", label="4 clinicians (no AI)")

    # finding first crossing point
    # crossing: y <= ref
    crossing_idx = np.where(y <= ref_value)[0]
    if len(crossing_idx) > 0:
        i = crossing_idx[0]
        plt.scatter([x[i]], [y[i]], s=80)
        plt.text(x[i], y[i], f"  break-even ≈ {x[i]:.0f}%", va="bottom")
    else:
        plt.text(
            x.iloc[0] if hasattr(x, "iloc") else x[0],
            max(y),
            "No break-even in range",
            va="bottom",
        )

    plt.xlabel("AI Effectiveness (%)")
    plt.ylabel(kpi_label)
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.show()


def plot_throughput_vs_eff(df_sweep, ref_value):
    x = df_sweep["eff"] * 100
    y = df_sweep["throughput"].values

    plt.figure(figsize=(6, 4))
    plt.plot(x, y, marker="o", label="3 clinicians + AI")
    plt.axhline(ref_value, linestyle="--", label="4 clinicians (no AI)")

    crossing_idx = np.where(y >= ref_value)[0]
    if len(crossing_idx) > 0:
        i = crossing_idx[0]
        plt.scatter([x[i]], [y[i]], s=80)
        plt.text(x[i], y[i], f"  break-even ≈ {x[i]:.0f}%", va="bottom")
    else:
        plt.text(x[0], max(y), "No break-even in range", va="bottom")

    plt.xlabel("AI Effectiveness (%)")
    plt.ylabel("Throughput per min")
    plt.title("Throughput vs AI Effectiveness")
    plt.grid(True)
    plt.legend()
    plt.show()

In [ ]:
# [13] necessary plots of kpis

plot_kpi_vs_eff(
    df_sweep=df_sweep_mc,
    ref_value=base_metrics["los_mean_ref"],
    kpi_col="los_mean",
    kpi_label="Mean LOS (min)",
    title="Mean LOS vs AI Effectiveness (3 clinicians + AI vs 4 clinicians)",
)

plot_kpi_vs_eff(
    df_sweep=df_sweep_mc,
    ref_value=base_metrics["los_p95_ref"],
    kpi_col="los_p95",
    kpi_label="LOS p95 (min)",
    title="LOS p95 vs AI Effectiveness",
)

plot_kpi_vs_eff(
    df_sweep=df_sweep_mc,
    ref_value=base_metrics["decision_latency_ref"],
    kpi_col="decision_latency_mean",
    kpi_label="Decision Latency Mean (min)",
    title="Decision Latency vs AI Effectiveness",
)

plot_throughput_vs_eff(df_sweep_mc, base_metrics["throughput_ref"])


In [ ]:
# [14] Sensitivity analysis – LOS and Throughput vs inter-arrival time
#
# The arrays below are pre-computed outputs from run_scenario() with Monte Carlo
# (N_REP = 1000) across six inter-arrival time values:
#   x = [3.2, 3.3, 3.4, 3.9, 4.0, 4.1]  (minutes)
#
# For each x, two configurations were compared:
#   - Baseline:    4 clinicians, ai_enabled=False
#   - AI-assisted: 3 clinicians, ai_enabled=True, AI_EFFECTIVENESS=0.25
#
# All other parameters held at BASE_PARAMS defaults.
# Results (mean and std over 1000 reps) were copied here to avoid re-running
# the full sweep every time the plotting cells are executed.
#
# To reproduce: loop run_scenario() over the x values above with the same seed
# strategy used in cell [10], collect los_mean/std and throughput mean/std,
# and replace the arrays below.

import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# X axis: inter-arrival times
# -----------------------------
x = np.array([3.2, 3.3, 3.4, 3.9, 4.0, 4.1])
offset = 0.03
width = 0.05

# -----------------------------
# LOS data (example)
# -----------------------------
los_base_mean = np.array([65.00, 56.53, 50.03, 34.82, 34.20, 33.09])
los_base_std = np.array([36.09, 32.45, 28.21, 8.940, 8.090, 4.150])

los_ai_mean = np.array([60.23, 52.49, 44.50, 31.18, 30.41, 29.89])
los_ai_std = np.array([35.56, 31.20, 23.96, 6.140, 4.310, 3.560])

# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(8, 5))

# Baseline candles
plt.bar(
    x - offset,
    2 * los_base_std,
    bottom=los_base_mean - los_base_std,
    width=width,
    alpha=0.6,
    label="4 clinicians (baseline)",
)

# AI candles
plt.bar(
    x + offset,
    2 * los_ai_std,
    bottom=los_ai_mean - los_ai_std,
    width=width,
    alpha=0.6,
    label="3 clinicians + AI (25%)",
)

# Mean markers
plt.scatter(x - offset, los_base_mean, color="black", s=25)
plt.scatter(x + offset, los_ai_mean, color="black", s=25)

plt.xlabel("Inter-arrival time (min)")
plt.ylabel("LOS (min)")
plt.title("Sensitivity Analysis – LOS (Mean ± Std, AI = 25%)")
plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# Throughput data (example)
# -----------------------------
thr_base_mean = np.array([0.2198, 0.2229, 0.2234, 0.2076, 0.2029, 0.1978])
thr_base_std = np.array([0.0252, 0.0224, 0.0204, 0.0173, 0.0175, 0.0173])

thr_ai_mean = np.array([0.2443, 0.2294, 0.2267, 0.2068, 0.2021, 0.1974])
thr_ai_std = np.array([0.0182, 0.0192, 0.0184, 0.0169, 0.0169, 0.0171])

plt.figure(figsize=(8, 5))

plt.bar(
    x - offset,
    2 * thr_base_std,
    bottom=thr_base_mean - thr_base_std,
    width=width,
    alpha=0.6,
    label="4 clinicians (baseline)",
)

plt.bar(
    x + offset,
    2 * thr_ai_std,
    bottom=thr_ai_mean - thr_ai_std,
    width=width,
    alpha=0.6,
    label="3 clinicians + AI (25%)",
)

plt.scatter(x - offset, thr_base_mean, color="black", s=25)
plt.scatter(x + offset, thr_ai_mean, color="black", s=25)

plt.xlabel("Inter-arrival time (min)")
plt.ylabel("Throughput (patients/min)")
plt.title("Sensitivity Analysis – Throughput (Mean ± Std, AI = 25%)")
plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
